In [1]:
from BCEmbedding.tools.langchain import BCERerank

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import UnstructuredPDFLoader
# FAISS是一个高效的相似性搜索库，常用于大规模向量搜索。
from langchain_community.vectorstores.faiss import FAISS

In [3]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain.retrievers import ContextualCompressionRetriever

In [4]:
from pprint import pprint

In [5]:
# 初始化embedding模型

embedding_model_name = "/root/project/models/bce-embedding-base_v1"
embedding_model_kwargs = {"device": "cuda:0"}
embedding_encode_kwargs = {"batch_size": 32,
                           "normalize_embeddings": True,
                          }

In [6]:
embed_model = HuggingFaceEmbeddings(
    model_name = embedding_model_name,
    model_kwargs = embedding_model_kwargs,
    encode_kwargs = embedding_encode_kwargs
)

03/18/2024 01:40:06 - [INFO] -sentence_transformers.SentenceTransformer->>>    Load pretrained SentenceTransformer: /root/project/models/bce-embedding-base_v1


In [7]:
# 初始化rereanker模型

reranker_args = {
    "model": "/root/project/models/bce-reranker-base_v1/",
    "top_n": 5,
    "device": "cuda:0"
}

reranker = BCERerank(**reranker_args)

03/18/2024 01:40:10 - [INFO] -BCEmbedding.models.RerankerModel->>>    Loading from `/root/project/models/bce-reranker-base_v1/`.
03/18/2024 01:40:10 - [INFO] -BCEmbedding.models.RerankerModel->>>    Execute device: cuda:0;	 gpu num: 1;	 use fp16: False


In [8]:
# 文档数据预处理

documents = UnstructuredPDFLoader("files/test.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,
                                               chunk_overlap=200)

texts = text_splitter.split_documents(documents)

03/18/2024 01:40:23 - [INFO] -pikepdf._core->>>    pikepdf C++ to Python logger bridge initialized


In [9]:
print(len(texts))

25


In [10]:
pprint(texts[0])

Document(page_content='互联网创作者经济白皮书\n\n©2022.8 iResearch Inc.\n\n摘要\n\n创作者经济可以理解为热情经济的子集，更加聚焦于互联网平台内容，指互联网内容创作者在创作工具、内容分发平台 及一系列其他创作者相关服务的帮助下产生经济收益。\n\nSM S\n\nWeb 1.0时代的内容创作特征：内容传播具有单向性，以平台创造、平台所有为主要特征；早期数字内容创作门槛高， 创作者规模小，以专业人群为主。\n\n随着用户创作需求的增长，用户已不单单满足于工具的使用，同时对文档、PPT、海报的模板和创意素材资源及其他相 关服务的需求也不断增加，催生出一批提供相关专业服务的平台。\n\nWeb 2.0网站和应用的发展成熟让更多用户参与到互联网内容创作中，内容平台及创作工具赋能下，互联网内容创作 者规模持续增长。\n\n处理数字图像、音频、视频等内容的数字创意类工具伴随计算机的诞生和互联网的发展而持续迭代，在PC及移动端形 成互为补充、多端互联的发展趋势。\n\n全球范围内，头部企业及资本已展开针对创作者经济的多方布局，部分新兴服务及产品亦初露头角；Web 3.0关键技 术发展驱动虚拟内容生产及消费，用户对3D、虚拟内容创作工具及服务的需求将进一步增长。\n\n来源：艾瑞咨询研究院自主研究及绘制。\n\n©2022.8 iResearch Inc.\n\nwww.iresearch.com.cn\n\n2\n\n定义及研究范畴 创作者经济是什么？\n\n本报告中的创作者一词指的是生产、拥有互联网内容，并向其他受众分发自己创作的内容的人。这些内容以文字、图片、播客、音频、视频、游戏等多种数字形式\n\n呈现。\n\n创作者经济可以理解为热情经济的子集，更加聚焦于互联网平台内容，指互联网内容创作者在创作工具、内容分发平台及一系列其他创作者相关服务的帮助下产生 经济收益。当前阶段，互联网内容创作者主要通过社交媒体等平台分发自己创作的内容，积累粉丝，并通过品牌赞助、广告分成、付费订阅等模式获得收益。\n\n零工经济与热情经济概念对比\n\n创作者经济关键词\n\n零工经济 TheGigEconomy\n\n热情经济 ThePassionEconomy\n\n工作举例\n\n滴滴司机、外卖员、接单设计师等\n\n作家、独立软件开发

In [11]:
# FAISS.from_documents(...) 从给定的文档集合texts中创建一个FAISS索引

# distrance_strategy : 指定了距离计算策略为最大内积（MAX_INNER_PRODUCT）。
# 在相似性搜索中，内积常用于衡量向量之间的相似度，最大内积策略意味着会优先考虑内积值最大的项，即最相似的文档。

# as_retriever() : 将之前创建的FAISS索引转化为一个检索器，用于执行相似性搜索。
# score_threshold=0.3表示只有当相似性得分超过0.3时，结果才会被认为是相关的，这是一个过滤结果的阈值。
# "k": 10指定了返回结果的数量，即对于每个查询，返回得分最高的10个文档。

retriever = FAISS.from_documents(texts, 
                                 embed_model, 
                                 distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
                                ).as_retriever(search_type="similarity", 
                                               search_kwargs={"score_threshold": 0.3, "k": 10})

03/18/2024 01:40:29 - [INFO] -faiss.loader->>>    Loading faiss with AVX2 support.
03/18/2024 01:40:29 - [INFO] -faiss.loader->>>    Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
03/18/2024 01:40:29 - [INFO] -faiss.loader->>>    Loading faiss.
03/18/2024 01:40:29 - [INFO] -faiss.loader->>>    Successfully loaded faiss.


In [12]:
# 首先使用retriever来进行初始的文档检索，然后可能使用base_compressor（即reranker）来进一步处理这些检索结果，
# 比如通过重新排序来提高检索结果的相关性。

compression_retriever = ContextualCompressionRetriever(
    base_compressor = reranker,
    base_retriever = retriever,
)

In [13]:
response = compression_retriever.get_relevant_documents("万兴科技做什么的?")

Token indices sequence length is longer than the specified maximum sequence length for this model (771 > 512). Running this sequence through the model will result in indexing errors
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [14]:
print(response[0].page_content)

来源：Adobe，及其他公开资料，艾瑞咨询研究院自主研究及绘制。

©2022.8 iResearch Inc.

www.iresearch.com.cn

32

创 作 者

平 台

工 具 及 服 务

互联网内容创作工具及服务典型企业案例 万兴科技：中国数字创意软件龙头，以创作者为中心，创意工具与资源并驾齐驱

万兴科技是全球领先的新生代数字创意赋能者，成立于2003年，面向全球海量新生代互联网用户，提供简单高效的数字创意软件、潮流时尚的创意资源和丰富多元 的生态化服务，旗下产品覆盖视频创意（万兴喵影、Wondershare Filmora、Wondershare VidAir、StoryChic、VidChic、万兴优转、万兴录演等）、绘图创意 （亿图图示、亿图脑图MindMaster、墨刀等）、文档创意（万兴PDF、Wondershare PDFelement、Wondershare Document Cloud等）、实用工具（万兴恢 复专家、万兴数据管家、万兴易修、Wondershare Dr.Fone等）等多个领域，业务范围遍及全球 200 多个国家和地区，用户活跃数超1亿，累计用户数超15亿。

布局全栈式视频创意工具，围绕创作者提供生产工具、创意资源及培训赋能等

视 频 创 意

万兴 喵影

万兴 优转

Beat.ly

StoryChic

Sweet Selfie

……

桌

面

创作者能力扶持

创作基地建设

Sandbox 万兴科技元宇宙 创作者俱乐部

端

+

创作者

绘 图 创 意

亿图 图示

亿图脑图 MindMaster

墨刀

Pixso

……

W

e

b

+

创意资源平台建设

移

结合AI等智能化技术，实现素

文 档 创 意

万兴 PDF

Wondershare PDF Converter

……

动

端

生产工具

创意资源

材智能识别、标记等功能，深

化全球全链条正版素材库布局，

打造坚实的内容底层生态平台，

建立“数字原料工厂”

来源：万兴科技，及其他公开资料，艾瑞咨询研究院自主研究及绘制。

©2022.8 iResearch Inc.

www.iresearch.com.cn

33

创 作 者

平 台

工 具 及 服 务

互联网内容创作工